# Match orphan status

In [4]:
import pandas as pd
import xlrd

In [6]:
xls_file = "./../data/FDA/Orphan_Drug_Status_FDA.xls"
df_orphan = pd.read_excel(xls_file, sheet_name=0)
df_orphan.head(2)


,Generic Name,Trade Name,Date Designated,Orphan Designation,Orphan Designation Status,Date Designation Withdrawn or Revoked,FDA Orphan Approval Status,Approved Labeled Indication,Marketing Approval Date,Exclusivity End Date,"Exclusivity Protected Indication * (Shown for approvals from Jan. 1, 2013, to the present)",Sponsor Company,Sponsor Address 1,Sponsor Address 2,Sponsor City,Sponsor State,Sponsor Zip,Sponsor Country,CF Grid Key
0,bosentan,Tracleer,2000-06-10 00:00:00,Treatment of pulmonary arterial hypertension,Designated/Approved,NaN,NaN,Treatment of pulmonary arterial hypertension (...,2017-05-09 00:00:00,2024-05-09 00:00:00,Treatment of pulmonary arterial hypertension (...,Actelion Pharmaceuticals Ltd,1840 Gateway Drive,Suite 300,Cherry Hill,New Jersey,' 08002 ',United States,134200.0
1,bosentan,Tracleer,2000-06-10 00:00:00,Treatment of pulmonary arterial hypertension,Designated/Approved,NaN,NaN,Treatment of pulmonary arterial hypertension.,11/20/2001,11/20/2008,NaN,Actelion Pharmaceuticals Ltd,1840 Gateway Drive,Suite 300,Cherry Hill,New Jersey,' 08002 ',United States,134200.0


In [10]:
fda_main = "./../data/FDA/formatted_output_openFDA.json"
df_fda_main = pd.read_json(fda_main)
df_fda_main.head(2)

,Origin,Marketing authorisation Number,Drug name,Non proprietary name,Marketing authorisation holder/ applicant,Pharmaceutical form,Administration route,Decision,Decision date,Current status,Non-clinical abridge,Referral
0,FDA,ANDA076177,CAMILA,NORETHINDRONE,DR REDDYS LABS SA,TABLET,ORAL-28,Approved,21.10.2002,authorised,yes,No
1,FDA,ANDA076178,NIZATIDINE,NIZATIDINE,EPIC PHARMA LLC,CAPSULE,ORAL,Approved,05.07.2002,authorised,yes,No


In [26]:
# Pre-lowercase the comparison column (dropna for safety)
fda_names_lower = df_fda_main['Drug name'].dropna().str.lower()

matched_rows = []
non_matched_rows = []
counter_matched = 0
counter_non_matched = 0

for _, arow in df_orphan.iterrows():
    trade_name = arow.get('Trade Name', '')

    if isinstance(trade_name, str):
        trade_name_lower = trade_name.lower()
        if trade_name_lower in fda_names_lower.values:
            matched_rows.append(arow)
            counter_matched += 1
        else:
            non_matched_rows.append(arow)
            counter_non_matched += 1
    else:
        non_matched_rows.append(arow)

# Convert matched and non-matched rows to DataFrames
df_matched = pd.DataFrame(matched_rows)
df_non_matched = pd.DataFrame(non_matched_rows)

print(f"Matched rows: {counter_matched}")
print(f"Non-matched rows: {counter_non_matched}")

# Save the results to CSV files
df_matched.to_csv('./../data/FDA/orphan_drugs_matched.csv', index=False)
df_non_matched.to_csv('./../data/FDA/orphan_drugs_non_matched.csv', index=False)

df_non_matched.head(2)


Matched rows: 993
Non-matched rows: 285


,Generic Name,Trade Name,Date Designated,Orphan Designation,Orphan Designation Status,Date Designation Withdrawn or Revoked,FDA Orphan Approval Status,Approved Labeled Indication,Marketing Approval Date,Exclusivity End Date,"Exclusivity Protected Indication * (Shown for approvals from Jan. 1, 2013, to the present)",Sponsor Company,Sponsor Address 1,Sponsor Address 2,Sponsor City,Sponsor State,Sponsor Zip,Sponsor Country,CF Grid Key
2,(tisagenlecleucel) Autologous T Cells transduc...,Kymriah (tisagenlecleucel),01/31/2014,For the treatment of Acute Lymphoblastic Leukemia,Designated/Approved,NaT,NaN,Treatment of patients up to 25 years of age wi...,08/30/2017,08/30/2024,Treatment of patients up to 25 years of age wi...,Novartis Pharmaceuticals Corporation,"One Health Plaza,",Bldg 315 - Room 3650B,East Hanover,New Jersey,' 07936 ',United States,415113.0
5,acalabrutinib,NaN,05/13/2015,Treatment of chronic lymphocytic leukemia (CLL).,Designated/Approved,NaT,NaN,CALQUENCE is indicated for the treatment of ad...,11/21/2019,11/21/2026,Indicated for the treatment of adult patients ...,"Acerta Pharma, LLC (a member of the AstraZenec...",121 Oyster Point Boulevard,NaN,South San Francisco,California,' 94080 ',United States,477415.0
